# **USING OBJECT DETECTION TO DETECT PEST**

## **Set up**

In [1]:
import torch
import cv2
import albumentations as A
import zipfile
from ultralytics import YOLO

import sys
from pathlib import Path

TRAINING_ROOT = Path().resolve().parent
sys.path.append(str(TRAINING_ROOT))

from config.config import (
    DATA_DIR, CONFIG_DIR, MODEL_DIR,
    DATASET_ZIP, DATASET_DIR,
    TRAIN_CONFIG, N_AUGMENT, CLASS_NAMES
)
from utils.util import unzip_files, verify_dataset

CONFIG = CONFIG_DIR / 'data.yaml'
OUTPUT = MODEL_DIR  / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)

## **Check GPU**

In [2]:
print(f'CUDA Available  : {torch.cuda.is_available()}')
print(f'CUDA Count      : {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'Device {i}     : {torch.cuda.get_device_name(i)}')

if torch.cuda.is_available():
    DEVICE = 0
    print(f'\nUsing GPU: {torch.cuda.get_device_name(DEVICE)}')
else:
    DEVICE = 'cpu'
    print('\nWARNING!!! CUDA not available - falling back to CPU.')

CUDA Available  : True
CUDA Count      : 1
Device 0     : NVIDIA GeForce RTX 5060 Laptop GPU

Using GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## **Unzip Dataset**

In [3]:
unzip_files(DATASET_ZIP, DATASET_DIR)
verify_dataset(DATASET_DIR)

D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\data\dataset already exists. Skipping unzip.
Verifying...
OK train --> images:  2607 | labels:  2553
OK val   --> images:   340 | labels:   316
OK test  --> images:   170 | labels:   157

Dataset is valid.


## **Augmentation**

In [4]:
augment_pipeline = A.Compose([
    # Geometry
    A.HorizontalFlip(p=0.5),
    A.Perspective(scale=(0.05, 0.1), p=0.3),
    # Color/Lighting
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=30, val_shift_limit=20, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.3),
    A.RandomShadow(shadow_roi=(0, 0.5, 1, 1), p=0.2),
    # Noise/Blur
    A.GaussNoise(std_range=(0.01, 0.05), p=0.5),
    A.GaussianBlur(blur_limit=3, p=0.3),
    # Regularization
    A.CoarseDropout(
        num_holes_range=(2, 6),
        hole_height_range=(10, 40),
        hole_width_range=(10, 40),
        p=0.3
    ),
], bbox_params=A.BboxParams(
    format='yolo',
    label_fields=['class_labels'],
    min_visibility=0.3,
    clip=True
))


def read_label(path):
    boxes, labels = [], []
    if not Path(path).exists():
        return boxes, labels
    with open(path, 'r') as f:
        for line in f:
            parts = list(map(float, line.strip().split()))
            if len(parts) == 5:
                cls, x, y, w, h = parts
                x = max(0.0, min(1.0, x))
                y = max(0.0, min(1.0, y))
                w = max(0.0, min(1.0, w))
                h = max(0.0, min(1.0, h))
                boxes.append([x, y, w, h])
                labels.append(int(cls))
    return boxes, labels


def save_label(path, boxes, labels):
    with open(path, 'w') as f:
        for box, cls in zip(boxes, labels):
            f.write(f'{cls} {box[0]:.6f} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f}\n')


def augment_dataset(img_dir, lbl_dir, n_augment=N_AUGMENT):
    img_dir = Path(img_dir)
    lbl_dir = Path(lbl_dir)

    # Remove old augmented files
    old_files = list(img_dir.glob('*_aug*.jpg')) + list(lbl_dir.glob('*_aug*.txt'))
    for f in old_files:
        f.unlink()
    print(f'Removed {len(old_files)} old augmented files.')

    img_files = list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png'))
    print(f'Augmenting {n_augment}x on {len(img_files)} images…')

    created = 0
    for img_path in img_files:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        boxes, labels = read_label(str(lbl_dir / (img_path.stem + '.txt')))
        if not boxes:
            continue
        for i in range(n_augment):
            try:
                result = augment_pipeline(image=img, bboxes=boxes, class_labels=labels)
                if not result['bboxes']:
                    continue
                save_name = f'{img_path.stem}_aug{i}'
                cv2.imwrite(str(img_dir / f'{save_name}.jpg'), result['image'])
                save_label(str(lbl_dir / f'{save_name}.txt'), result['bboxes'], result['class_labels'])
                created += 1
            except Exception as e:
                print(f'Error {img_path.name}: {e}')

    orig = len(img_files)
    print(f'DONE! Original: {orig} | Added: {created} | Total: {orig + created}')


augment_dataset(
    img_dir=DATASET_DIR / 'images' / 'train',
    lbl_dir=DATASET_DIR / 'labels' / 'train',
)

Removed 3814 old augmented files.
Augmenting 3x on 700 images…


d:\IT\Computer Vision\Object Detection\pest_detection\venv1\Lib\site-packages\albumentations\augmentations\dropout\functional.py:559: RuntimeWarning: invalid value encountered in divide
  visibility_ratios = remaining_areas / box_areas


DONE! Original: 700 | Added: 1904 | Total: 2604


## **Train YOLOv8**

In [5]:
model = YOLO(TRAIN_CONFIG['model'])

results = model.train(
    data          = str(CONFIG),
    project       = str(OUTPUT),
    name          = 'run2',
    device        = DEVICE,
    epochs        = TRAIN_CONFIG['epochs'],
    imgsz         = TRAIN_CONFIG['imgsz'],
    batch         = TRAIN_CONFIG['batch'],
    workers       = TRAIN_CONFIG['workers'],
    patience      = TRAIN_CONFIG['patience'],
    optimizer     = TRAIN_CONFIG['optimizer'],
    lr0           = TRAIN_CONFIG['lr0'],
    lrf           = TRAIN_CONFIG['lrf'],
    weight_decay  = TRAIN_CONFIG['weight_decay'],
    cos_lr        = TRAIN_CONFIG['cos_lr'],
    warmup_epochs = TRAIN_CONFIG['warmup_epochs'],
    augment       = TRAIN_CONFIG['augment'],
    cache         = TRAIN_CONFIG['cache'],
    save          = TRAIN_CONFIG['save'],
    plots         = TRAIN_CONFIG['plots'],
    verbose       = TRAIN_CONFIG['verbose'],
    exist_ok      = TRAIN_CONFIG['exist_ok'],
)

BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'
print(f'\nBest model saved at: {BEST_MODEL_PATH}')

Ultralytics 8.4.52  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\config\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=run2, n

## **Resume Training**

In [6]:
# model = YOLO(str(OUTPUT / 'run1/weights/last.pt'))
# results = model.train(resume=True)
# BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'

## **Evaluate on Validation**

In [6]:
best_model = YOLO(str(BEST_MODEL_PATH))
v_metrics  = best_model.val(data=str(CONFIG), split='val', device=DEVICE)

print("===== RESULT ON VALIDATION =====")
print(f"mAP@0.5     : {v_metrics.box.map50:.4f}  ({v_metrics.box.map50*100:.1f}%)")
print(f"mAP@0.5:0.95: {v_metrics.box.map:.4f}  ({v_metrics.box.map*100:.1f}%)")
print(f"Precision   : {v_metrics.box.p.mean():.4f}")
print(f"Recall      : {v_metrics.box.r.mean():.4f}")
print()
for i, c in enumerate(CLASS_NAMES):
    try:
        ap = v_metrics.box.ap[i]
        print(f"{c:12s}: AP@0.5:0.95 = {ap:.4f} ({ap*100:.1f}%)")
    except:
        pass

Ultralytics 8.4.52  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
Model summary (fused): 93 layers, 25,841,497 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 1146.0442.0 MB/s, size: 64.8 KB)
val: Scanning D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\data\dataset\labels\val.cache... 316 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 340/340  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 4.7it/s 4.7s0.2s
                   all        340       2963      0.722      0.741      0.774      0.459
                 bo to         28         32      0.689      0.875      0.877      0.665
                bo map        260       2028      0.781      0.787      0.812       0.38
               bo thon        238        903      0.698      0.561      0.633      0.331
Speed: 2.1ms preprocess, 8.8ms infer

## **Evaluate on Test**

In [7]:
t_metrics = best_model.val(data=str(CONFIG), split='test', device=DEVICE)

print('===== RESULT ON TEST =====')
print(f"mAP@0.5     : {t_metrics.box.map50:.4f}  ({t_metrics.box.map50*100:.1f}%)")
print(f"mAP@0.5:0.95: {t_metrics.box.map:.4f}  ({t_metrics.box.map*100:.1f}%)")
print(f"Precision   : {t_metrics.box.p.mean():.4f}")
print(f"Recall      : {t_metrics.box.r.mean():.4f}")
print()
for i, c in enumerate(CLASS_NAMES):
    try:
        ap = t_metrics.box.ap[i]
        print(f'{c:12s}: AP@0.5:0.95 = {ap:.4f} ({ap*100:.1f}%)')
    except:
        pass

Ultralytics 8.4.52  Python-3.12.7 torch-2.12.0.dev20260408+cu128 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
val: Fast image access  (ping: 0.10.0 ms, read: 151.439.6 MB/s, size: 66.2 KB)
val: Scanning D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\data\dataset\labels\test.cache... 157 images, 14 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 170/170  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 5.1it/s 2.2s0.2s
                   all        170       1700       0.74       0.84      0.814       0.43
                 bo to         14         18      0.781          1      0.928      0.549
                bo map        127        926      0.778      0.853      0.876      0.454
               bo thon        104        756       0.66      0.668      0.638      0.286
Speed: 1.4ms preprocess, 8.4ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\ASUS\Downlo

## **Zip Results**

In [8]:
run1_dir = Path(results.save_dir)
zip_path = run1_dir.parent / 'run2.zip'

print(f'Zipping {run1_dir} …')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file in run1_dir.rglob('*'):
        if file.is_file():
            zf.write(file, file.relative_to(run1_dir.parent))

print(f'Done! Saved to: {zip_path}')
print(f'File size: {zip_path.stat().st_size / 1024 / 1024:.1f} MB')

Zipping D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\models\runs\run2 …
Done! Saved to: D:\IT\Computer Vision\Object Detection\pest_detection\pest_detection_training\models\runs\run2.zip
File size: 95.6 MB
